<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m7_03_nested_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M7.03 — Tuning with the wrong referee

In the lab we saw that a hyperparameter search contains a
cross-validation of its own, and that its score is too optimistic to be
reported: an outer cross-validation is needed around it (**nested
cross-validation**). In this workbook you tune a model on the
Parkinson's recordings and discover a second danger: when the **inner**
cross-validation asks the wrong question, the search does not only
report a wrong score, it chooses the **wrong model**.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "better_model_for_new_patients": "f7f74ccfe5e990d8",
    "grouped_search_prefers": "812b209fd726dcdb",
    "many_neighbours_becomes": "e03ea76792bff7fa",
    "misleading_search": "3b4461a31c14bdf6",
    "n_trainings_nested": "2811745d7b8d8874",
    "shuffled_search_prefers": "a7a3d1d1cb5a2922"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data, and the question we care about

5875 voice recordings of 42 patients; we predict `total_updrs`. From
workbook M7.02 we know that the question that matters to the hospital is
"how well does the model do for **patients it has never seen**?", and
that it is answered by keeping each patient's recordings together.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

recordings = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/parkinsons_telemonitoring.csv"
)
features = [
    name for name in recordings.columns if name not in ("subject", "test_time", "motor_updrs", "total_updrs")
]
data = recordings[features]
target = recordings["total_updrs"]
patient = recordings["subject"]

The model is a k-nearest-neighbours regressor: it predicts the average
score of the `n_neighbors` most similar recordings. We search the best
number of neighbours.

In [ ]:
model = make_pipeline(StandardScaler(), KNeighborsRegressor())
param_grid = {"kneighborsregressor__n_neighbors": [1, 3, 10, 30, 100, 300]}

## 2. A search with shuffled folds

First the way most tutorials do it: a grid search whose internal
cross-validation shuffles the recordings.

In [ ]:
shuffled_search = GridSearchCV(
    model, param_grid, cv=KFold(n_splits=5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error"
)
shuffled_search.fit(data, target)


def search_table(search):
    return pd.DataFrame(
        {"mean error in the search": -search.cv_results_["mean_test_score"]},
        index=pd.Index(param_grid["kneighborsregressor__n_neighbors"], name="n_neighbors"),
    )


print(f"chosen number of neighbours: {shuffled_search.best_params_['kneighborsregressor__n_neighbors']}")
print(f"error announced by the search (best_score_): {-shuffled_search.best_score_:.2f}")
search_table(shuffled_search).round(2)

The search is confident: a handful of neighbours, and an error around
4 points, half that of the baseline (8.9).

🧠 **Think.** With very few neighbours, whose recordings are the nearest
neighbours of a recording likely to be? Remember what `age` and `sex`
do in this dataset.

## 3. The same search, with patients kept together

`GridSearchCV` accepts groups too: we pass `groups=patient` to `fit`,
and use a `GroupKFold` as its internal cross-validation.

🔮 **Predict first.** When the search has to do well on patients it has
not seen, will it choose **"fewer neighbours"**, **"the same number"**
or **"many more neighbours"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["fewer neighbours", "the same number", "many more neighbours"])

In [ ]:
grouped_search = GridSearchCV(model, param_grid, cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error")
grouped_search.fit(data, target, groups=patient)

comparison = pd.concat(
    {"shuffled folds": search_table(shuffled_search), "patients kept together": search_table(grouped_search)},
    axis=1,
).droplevel(1, axis=1)
print(f"chosen number of neighbours: {grouped_search.best_params_['kneighborsregressor__n_neighbors']}")
ax = comparison.plot(marker="o", logx=True, figsize=(7, 3.5))
ax.set(xlabel="n_neighbors (log scale)", ylabel="mean error in the search (UPDRS points)")
comparison.round(2)

✏️ Which end of the grid does each search prefer? Write `"few
neighbours"` or `"many neighbours"` for each.

In [ ]:
shuffled_search_prefers = ____
grouped_search_prefers = ____
check("shuffled_search_prefers", shuffled_search_prefers)
check("grouped_search_prefers", grouped_search_prefers)

The two curves go in **opposite directions**. With shuffled folds, the
best strategy is to find the other recordings of the same patient: few
neighbours. For a new patient there are no such recordings, and those
few neighbours are other people who merely have the same age: the more
neighbours are averaged, the better.

The inner cross-validation is the referee of the search. With the wrong
referee, the search optimises the wrong game.

## 4. Nested cross-validation: what each search is really worth

To know how each tuned model does on new patients, we wrap each search
in an **outer** cross-validation that holds out whole patients. The
search runs again inside every outer training part, and never sees the
outer test patients.

✏️ The outer cross-validation has 6 folds. Inside each of them, a search
tries 6 values on 5 inner folds and then retrains the best one. How many
times is the model trained in total, for one nested cross-validation?

In [ ]:
n_trainings_nested = ____
check("n_trainings_nested", n_trainings_nested, hint="For one outer fold: 6 × 5 trainings and 1 refit. Then 6 outer folds.")

In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error

outer = GroupKFold(n_splits=6)
nested = {}
for name, search in [("shuffled folds", shuffled_search), ("patients kept together", grouped_search)]:
    fold_errors, chosen = [], []
    for train_rows, test_rows in outer.split(data, target, patient):
        inner_search = clone(search)
        if name == "patients kept together":
            inner_search.fit(data.iloc[train_rows], target.iloc[train_rows], groups=patient.iloc[train_rows])
        else:
            inner_search.fit(data.iloc[train_rows], target.iloc[train_rows])
        fold_errors.append(mean_absolute_error(target.iloc[test_rows], inner_search.predict(data.iloc[test_rows])))
        chosen.append(inner_search.best_params_["kneighborsregressor__n_neighbors"])
    nested[name] = {
        "error announced by the search": -search.best_score_,
        "error on new patients (outer folds)": np.mean(fold_errors),
        "neighbours chosen in the 6 outer folds": chosen,
    }
nested = pd.DataFrame(nested).T
numeric = ["error announced by the search", "error on new patients (outer folds)"]
nested[numeric] = nested[numeric].astype(float).round(2)
nested

✏️ For which search is the announced error far from the error measured
on new patients? And which search gives the better model for new
patients? Write `"shuffled folds"` or `"patients kept together"` for
each.

In [ ]:
misleading_search = ____
better_model_for_new_patients = ____
check("misleading_search", misleading_search)
check("better_model_for_new_patients", better_model_for_new_patients)

Two different damages, both caused by the shuffled inner folds:

- the **score** it announced (about 4) was far from the truth (about
  11);
- the **model** it chose is worse for new patients than the one chosen
  by the grouped search.

The outer loop of a nested cross-validation repairs the first damage: it
reports an honest score whatever the search does. It cannot repair the
second: the search has already picked its model. **Both loops must cut
the data the way the model will be used.**

### ➕ Going further: the edge of the grid

The grouped search chose 300 neighbours, the largest value we offered.
You know from workbook M3.01 what that means.

In [ ]:
wider_grid = {"kneighborsregressor__n_neighbors": [300, 1000, 2000, 4000]}
wider_search = GridSearchCV(model, wider_grid, cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error")
wider_search.fit(data, target, groups=patient)
pd.DataFrame(
    {"mean error in the search": -wider_search.cv_results_["mean_test_score"]},
    index=pd.Index(wider_grid["kneighborsregressor__n_neighbors"], name="n_neighbors"),
).round(2)

✏️ A k-nearest-neighbours model that averages thousands of recordings
predicts almost the same value for everybody. Which model of workbook
M7.01 is it turning into? Write `"the dummy"` or `"gradient boosting"`.

In [ ]:
many_neighbours_becomes = ____
check("many_neighbours_becomes", many_neighbours_becomes)

An honest search, given enough room, walks towards "predict the mean
for everyone". It is the search's way of saying what workbook M7.02
found: with these features, there is nothing reliable to learn about a
new patient. A hyperparameter search cannot create information; it can
only stop you from pretending you have some.

🧠 **Think.**

1. In module 3 we used nested cross-validation to get an honest score.
   What is the new lesson of this workbook about the **inner** loop?
2. A colleague tunes a model for next month's sales with shuffled folds
   over three years of daily data. Which mistake of this workbook is
   being repeated, and what would a correct inner split look like?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the two curves of the search: why shuffled folds prefer few
     neighbours and grouped folds many;
   - what the nested cross-validation showed about each search: the
     announced score and the real one;
   - why the outer loop cannot repair a wrong inner loop;
   - the wider grid: what the search is telling us when it keeps asking
     for more neighbours.